# Klasifikasi Tingkat Kemiskinan Kabupaten/Kota di Indonesia
### Perbandingan Algoritma Machine Learning untuk Mendukung SDGs 1: Tanpa Kemiskinan

Notebook ini disusun mengikuti **7 tahap pengembangan proyek AI**:
1. Problem &nbsp;→&nbsp; 2. Data &nbsp;→&nbsp; 3. Preprocessing &nbsp;→&nbsp; 4. Training
&nbsp;→&nbsp; 5. Evaluation &nbsp;→&nbsp; 6. Deployment &nbsp;→&nbsp; 7. Monitoring

Penomoran section di bawah sengaja dibuat persis mengikuti 7 tahap ini, supaya jelas bagian
mana yang termasuk preprocessing, training, evaluasi, dst.

## 1. Problem (Penetapan Masalah)

**Pertanyaan utama:** Bisakah status kemiskinan suatu kabupaten/kota (Miskin / Tidak Miskin)
diprediksi dari indikator sosial-ekonominya (pendidikan, pengeluaran, kesehatan, akses
infrastruktur, ketenagakerjaan, dan ukuran ekonomi daerah)?

**Kenapa penting:** kalau pola ini bisa dipelajari dari data yang sudah ada, model bisa
membantu mengidentifikasi wilayah mana yang berisiko/berstatus miskin berdasarkan indikator
yang lebih mudah dipantau, mendukung SDGs 1 (Tanpa Kemiskinan).

**Jenis masalah:** klasifikasi biner (dua kelas: Miskin = 1, Tidak Miskin = 0).

## 2. Data (Pengumpulan Data)

**Sumber:** dataset publik "Klasifikasi Tingkat Kemiskinan di Indonesia" dari Kaggle,
berisi 514 baris (1 baris = 1 kabupaten/kota), bersumber dari data BPS/Susenas.

**Fitur (input, X)** — 9 indikator sosial-ekonomi:
1. Rata-rata Lama Sekolah Penduduk 15+ (Tahun)
2. Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun)
3. Indeks Pembangunan Manusia (IPM)
4. Umur Harapan Hidup (Tahun)
5. Persentase rumah tangga dengan akses sanitasi layak
6. Persentase rumah tangga dengan akses air minum layak
7. Tingkat Pengangguran Terbuka (TPT)
8. Tingkat Partisipasi Angkatan Kerja (TPAK)
9. PDRB atas Dasar Harga Konstan menurut Pengeluaran (Rupiah)

(Kolom "Persentase Penduduk Miskin (P0)" sengaja **tidak** dimasukkan sebagai fitur akhir —
lihat alasannya di bagian Evaluation, sub-bab Data Leakage.)

**Label (target, y):** `Klasifikasi Kemiskinan` — 0 = Tidak Miskin, 1 = Miskin.

File CSV-nya berformat Eropa: kolom dipisah `;` dan desimal ditulis pakai koma
(`18,98` bukan `18.98`), sehingga saat membaca file perlu parameter `sep=';'` dan
`decimal=','`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE

df = pd.read_csv("Klasifikasi_Tingkat_Kemiskinan_di_Indonesia.csv", sep=';', decimal=',')
print("Ukuran data mentah:", df.shape)
df.head()

## 3. Preprocessing (Pembersihan dan Penyiapan Data)

Semua langkah menyiapkan data mentah menjadi siap dilatih dikumpulkan di satu bagian ini,
sesuai kerangka 7 tahap:

**a) Menangani missing value / baris kosong.**
File aslinya menyisakan ratusan baris kosong di bagian bawah (bekas template Excel) —
dibuang dengan `dropna(how='all')`.

**b) Mengubah format data.**
Kolom *Persentase Penduduk Miskin (P0)* dan *Rata-rata Lama Sekolah* terbaca sebagai teks
karena format desimal koma tidak terbaca otomatis oleh pandas — diperbaiki manual (ganti
koma jadi titik, lalu konversi ke `float`).

**c) Memisahkan fitur (X) dan label (y).**
Kolom `Provinsi` dan `Kab/Kota` dibuang dari X karena berupa teks/nama daerah, bukan angka
prediktor. Kolom `Persentase Penduduk Miskin (P0)` juga sengaja dibuang dari X (lihat
penjelasan data leakage di bagian Evaluation).

**d) Normalisasi/standarisasi.**
Skala 9 fitur sangat berbeda jauh — misalnya PDRB bisa bernilai jutaan-miliaran rupiah,
sementara IPM hanya berkisar 0–100. Untuk algoritma yang menghitung berdasarkan **jarak**
atau **koefisien** (KNN, SVM, Logistic Regression), perbedaan skala ini bisa membuat
variabel bernilai besar seperti PDRB "mendominasi" perhitungan meski belum tentu itu
variabel yang paling penting. Sebagai solusi, ketiga model itu dibungkus dengan
`StandardScaler` (dijalankan di bagian Training, lewat `make_pipeline`), sehingga semua
fitur diskalakan ke rentang yang sebanding (rata-rata 0, simpangan baku 1) sebelum masuk ke
model. Decision Tree, Random Forest, dan Naive Bayes tidak memerlukan ini karena cara kerja
mereka tidak bergantung pada skala angka.

**e) Menyeimbangkan kelas (SMOTE).**
Dari 514 data, hanya sekitar 12% berlabel "Miskin" — data tidak seimbang (*imbalanced*).
Kalau dibiarkan, model cenderung bias menebak kelas mayoritas terus dan tetap "terlihat"
akurat padahal gagal mengenali kelas minoritas. **SMOTE** membuat data sintetis untuk kelas
minoritas agar kedua kelas seimbang jumlahnya.

**f) Membagi data latih dan data uji (80:20).**
Supaya performa model diukur dari data yang belum pernah dilihat sebelumnya, sehingga
evaluasinya jujur.

In [ ]:
df = df.dropna(how='all').reset_index(drop=True)

col1 = 'Persentase Penduduk Miskin (P0) Menurut Kabupaten/Kota (Persen)'
col2 = 'Rata-rata Lama Sekolah Penduduk 15+ (Tahun)'
for c in [col1, col2]:
    df[c] = df[c].astype(str).str.replace(',', '.').astype(float)

print("Ukuran setelah baris kosong dibuang:", df.shape)

X = df.drop(columns=['Provinsi', 'Kab/Kota', 'Klasifikasi Kemiskinan', col1])
y = df['Klasifikasi Kemiskinan']
print("Jumlah fitur (X):", X.shape[1])
X.head()

In [ ]:
print("Distribusi kelas sebelum SMOTE:", y.value_counts().to_dict())

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)
print("Distribusi kelas sesudah SMOTE  :", y_resampled.value_counts().to_dict())

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)
print("\nData latih:", X_train.shape)
print("Data uji  :", X_test.shape)

## 4. Training (Melatih Model)

Enam algoritma klasifikasi dilatih dengan data yang sama, supaya perbandingannya adil:

| Algoritma | Perlu standarisasi? |
|---|---|
| Decision Tree | Tidak |
| Random Forest | Tidak |
| Naive Bayes | Tidak |
| KNN | **Ya** — dibungkus `StandardScaler` lewat `make_pipeline` |
| SVM | **Ya** — dibungkus `StandardScaler` lewat `make_pipeline` |
| Logistic Regression | **Ya** — dibungkus `StandardScaler` lewat `make_pipeline` |

`make_pipeline(StandardScaler(), model)` membuat scaler dan model jadi satu objek: setiap
kali `.fit()` dipanggil, scaler otomatis "belajar" skala data **hanya dari data latih**,
lalu skala itu juga dipakai saat `.predict()` pada data uji — jadi tidak ada kebocoran
informasi dari data uji ke proses standarisasi.

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=100),
    "Naive Bayes": GaussianNB(),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "SVM": make_pipeline(StandardScaler(), SVC(random_state=42, probability=True)),
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)),
}

results = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    predictions[name] = y_pred
    print(f"=== {name} ===")
    print(f"Akurasi: {acc:.4f}")
    print(classification_report(y_test, y_pred))
    print("-" * 50)

## 5. Evaluation (Mengevaluasi Model)

Bagian ini membandingkan performa semua model, memilih yang terbaik, lalu **mengecek apakah
performa itu valid** (bukan hasil kebocoran data).

**a) Perbandingan akurasi antar model** (tabel + grafik batang), lalu otomatis dipilih model
dengan akurasi tertinggi sebagai kandidat terbaik.

**b) Confusion Matrix** untuk model terbaik — merinci berapa banyak prediksi benar/salah per
kelas, bukan cuma satu angka akurasi.

**c) Feature Importance** — khusus model berbasis pohon, melihat variabel mana yang paling
menentukan keputusan model.

In [ ]:
results_df = pd.DataFrame(list(results.items()), columns=['Model', 'Akurasi'])
results_df = results_df.sort_values(by='Akurasi', ascending=False).reset_index(drop=True)
print(results_df)

plt.figure(figsize=(8, 5))
sns.barplot(data=results_df, x='Akurasi', y='Model', hue='Model', palette='viridis', legend=False)
plt.title("Perbandingan Akurasi Model")
plt.xlim(0, 1)
for i, v in enumerate(results_df['Akurasi']):
    plt.text(v + 0.01, i, f"{v:.4f}", va='center')
plt.show()

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]
best_pred = predictions[best_model_name]

print(f"Model terbaik: {best_model_name}")
print(f"Akurasi      : {results_df.iloc[0]['Akurasi']:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, best_pred))

cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f"Confusion Matrix - {best_model_name}")
plt.xlabel("Prediksi"); plt.ylabel("Aktual")
plt.show()

In [ ]:
tree_model = best_model.named_steps.get('decisiontreeclassifier') if hasattr(best_model, 'named_steps') else best_model
if hasattr(tree_model, 'feature_importances_'):
    importance_df = pd.DataFrame({
        'Fitur': X.columns,
        'Importance': tree_model.feature_importances_
    }).sort_values(by='Importance', ascending=False)

    plt.figure(figsize=(8, 5))
    sns.barplot(data=importance_df, x='Importance', y='Fitur', hue='Fitur', palette='mako', legend=False)
    plt.title(f"Feature Importance - {best_model_name}")
    plt.show()
    print(importance_df)
else:
    print(f"Model {best_model_name} tidak punya feature_importances_ bawaan (bukan model berbasis pohon).")

### 5a. Catatan Penting: Data Leakage yang Sudah Dihindari

Di awal pengerjaan, kolom **"Persentase Penduduk Miskin (P0)"** sempat ikut dijadikan fitur.
Hasilnya akurasi model (terutama Random Forest) melonjak sampai **98%** — angka yang
**mencurigakan**, bukan sesuatu yang langsung dianggap prestasi. Setelah ditelusuri, P0
kemungkinan besar adalah **variabel yang dipakai untuk menentukan label** `Klasifikasi
Kemiskinan` itu sendiri (mis. jika P0 di atas ambang tertentu, label = 1). Ini disebut
**data leakage**: model "mengintip" langsung salah satu sumber pembentuk labelnya, sehingga
akurasi tinggi di atas kertas tapi tidak benar-benar berguna untuk kasus baru di dunia
nyata.

**Buktinya** (dari eksperimen sebelumnya): setelah kolom P0 dibuang, akurasi turun
signifikan pada model berbasis pohon/linear (Random Forest turun dari 98,3% menjadi 91,2%,
Logistic Regression turun dari 97,2% menjadi sekitar 78%), sementara model KNN/SVM/Naive
Bayes tidak berubah karena dari awal tidak terlalu mengandalkan P0.

**Karena itu, di notebook versi ini kolom P0 sudah langsung dibuang dari fitur sejak bagian
Preprocessing** — supaya seluruh alur di atas (training, evaluasi, confusion matrix, feature
importance) sudah dihitung dari model yang jujur, tanpa perlu diulang dua kali.

## 6. Deployment (Menerapkan Model)

Secara sederhana, "deployment" berarti model yang sudah dilatih dipakai untuk memprediksi
data baru yang belum pernah dilihat sebelumnya — bukan lagi cuma data uji dari dataset asli.

Di bawah ini, model final (`best_model`, hasil terbaik dari bagian Evaluation) dipakai
lewat **input manual**: nilai 9 indikator suatu daerah dimasukkan langsung lewat keyboard,
lalu program menampilkan prediksi status kemiskinannya beserta tingkat keyakinan
(probabilitas) untuk tiap kelas.

Dalam penerapan nyata (di luar notebook), langkah berikutnya biasanya: model disimpan ke
file (`pickle`/`joblib`), lalu dihubungkan ke API (FastAPI/Flask) dan antarmuka web/mobile,
sehingga bisa dipakai pengguna lain tanpa harus membuka notebook ini.

In [ ]:
fitur_list = list(X.columns)
print("Fitur yang perlu diisi saat prediksi:")
for i, f in enumerate(fitur_list, 1):
    print(f"{i}. {f}")

print("\n=== Masukkan Data untuk Prediksi Kemiskinan ===\n")
input_values = {}
for fitur in fitur_list:
    while True:
        try:
            nilai = float(input(f"{fitur}: "))
            input_values[fitur] = nilai
            break
        except ValueError:
            print("   -> Input harus berupa angka, coba lagi.")

data_user = pd.DataFrame([input_values])[fitur_list]
prediksi = best_model.predict(data_user)
probabilitas = best_model.predict_proba(data_user)

label_map = {0: "Tidak Miskin", 1: "Miskin"}
print("\n=== HASIL PREDIKSI ===")
print(f"Status: {label_map[prediksi[0]]}")
print(f"Probabilitas Tidak Miskin: {probabilitas[0][0]:.2%}")
print(f"Probabilitas Miskin      : {probabilitas[0][1]:.2%}")

## 7. Monitoring (Memantau dan Memperbarui Model)

Model tidak boleh dianggap "selesai sekali dilatih". Setelah dipakai, beberapa hal perlu
dipantau secara berkala:

- **Pemantauan kinerja:** kalau data kabupaten/kota terbaru (misal rilis tahun berikutnya)
  sudah tersedia dan sudah diketahui label aslinya, bandingkan lagi prediksi model dengan
  kenyataan, hitung ulang akurasinya.
- **Data drift:** kondisi sosial-ekonomi daerah bisa berubah seiring waktu (kebijakan baru,
  krisis ekonomi, bencana). Kalau pola datanya sudah banyak berubah dari data training,
  model lama bisa jadi kurang akurat dan perlu dilatih ulang (*retraining*) dengan data
  terbaru.
- **Evaluasi berkala terhadap ambang batas label:** kalau definisi "Miskin/Tidak Miskin"
  yang dipakai BPS berubah, dataset dan label training juga perlu diperbarui.

## Kesimpulan

- **Model terbaik:** _isi nama model & akurasinya dari tabel `results_df` di bagian
  Evaluation_
- **Fitur paling berpengaruh:** _lihat hasil feature importance (jika model terbaik
  berbasis pohon)_
- **Kaitan dengan SDGs 1:** jelaskan bagaimana model ini bisa membantu mengidentifikasi
  wilayah prioritas untuk program pengentasan kemiskinan, berdasarkan variabel yang paling
  berpengaruh.